In [179]:
from collections import Counter

In [180]:
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

In [181]:
class BytePairEncoding:
    def __init__(self, num_merges):
        self.num_merges = num_merges
        self.vocab = []

    def _get_stats(self, ids):
        stats = Counter(zip(ids, ids[1:]))
        return sorted(stats.items(), key=lambda x: x[1], reverse=True)

    def _merge(self, ids, pair, new_id):
        new_ids = []
        i = 0
        while i < len(ids):
            if i + 1 < len(ids) and ids[i] == pair[0] and ids[i + 1] == pair[1]:
                new_ids.append(new_id)
                i += 1
            else:
                new_ids.append(ids[i])
            i += 1
        return new_ids

    def train(self, text):
        ids = list(text.encode("utf-8"))

        self.vocab = [(i,) for i in range(256)]

        for _ in range(self.num_merges):
            frequent_pair = self._get_stats(ids)[0][0]

            new_id = len(self.vocab)
            ids = self._merge(ids, frequent_pair, new_id)

            self.vocab.append(frequent_pair)

    def encode(self, text):
        ids = list(text.encode("utf-8"))

        i = 0
        while i < len(self.vocab):
            if len(self.vocab[i]) == 1:
                i += 1
                continue
            ids = self._merge(ids, self.vocab[i], i)
            i += 1

        return ids

    def decode(self, ids):
        for _ in range(self.num_merges):
            output = []
            i = 0
            while i < len(ids):
                output.extend(self.vocab[ids[i]])
                i += 1
            ids = output

        return bytes(ids).decode("utf-8", errors="replace")

In [182]:
bp = BytePairEncoding(num_merges=10)
bp.train(train)

In [183]:
bp.decode(bp.encode(val)) == val

True